# End-to-end pipeline (BPIC12 complete, remaining time)

1. Load the dataset already processed by the full experiment sweep (`experiments/full_sweep`).
2. Load it with `preprocessing.loader`.
3. Load the trained LSTM checkpoint from the same sweep (`models.lstm.model`).
4. Compute explainability (all 5 unified explainers) for one prefix, as pandas tables.
5. Run the sensitivity study (RIS/ROS/FA/RA, Monte Carlo) over a sample of prefixes, using
   Integrated Gradients, via `sensitivity_evaluation`.


In [1]:
import logging
import os
import sys
import warnings

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
warnings.filterwarnings("ignore")

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np
import pandas as pd
import tensorflow as tf

tf.get_logger().setLevel("ERROR")
logging.getLogger("alibi").setLevel(logging.ERROR)

from preprocessing.processor import Task
from preprocessing.loader import load_remaining_time
import models.lstm.model as model_module
from explainability.utils import build_column_blocks
from explainability.ig_explainer import build_ig_zero_baseline, explain_ig
from explainability.eg_explainer import build_eg_background, explain_eg
from explainability.gradient_saliency_explainer import explain_gradient_saliency
from explainability.shap_kernel_explainer import explain_shap
from explainability.lime_explanations import build_lime_layout, build_lime_background, explain_lime
from sensitivity_evaluation.sensitivity_perturbations import (
    extract_embedding, build_all_categorical_sampling_distributions,
)
from sensitivity_evaluation.metrics import relative_stability_montecarlo_v4_with_agreement

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

I0000 00:00:1795166021.262191  203503 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795166021.262724  203503 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1795166024.373447  203503 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795166024.373913  203503 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## 1. Dataset

BPIC12 complete, `lifecycle:transition == COMPLETE` events only. Features: `org:resource` (categorical, event-level), `case:AMOUNT_REQ` (numerical, case-level).

Already processed under `experiments/full_sweep` with this exact column configuration. Reused directly instead of reprocessing the raw CSV.

In [2]:
DATASET_NAME = "bpic12_complete_remaining_time"
FULL_SWEEP_PROCESSED_ROOT = os.path.join(REPO_ROOT, "processed_datasets", "full_sweep")
FULL_SWEEP_EXPERIMENTS_ROOT = os.path.join(REPO_ROOT, "experiments", "full_sweep")
PROCESSED_MULTI_DIR = os.path.join(FULL_SWEEP_PROCESSED_ROOT, DATASET_NAME, "processed_multi")

assert os.path.exists(os.path.join(PROCESSED_MULTI_DIR, "metadata.json")), (
    f"{PROCESSED_MULTI_DIR} does not exist -- expected the dataset already processed by "
    f"scripts/train_full_sweep.py"
)

## 2. Load with `preprocessing.loader`

In [3]:
data = load_remaining_time(PROCESSED_MULTI_DIR)

feature_names = data["feature_names"]
categorical_columns = data["categorical_columns"]
numerical_columns_list = data["numerical_columns"]
numerical_minmax = data["numerical_minmax"]
columns_info = data["columns_info"]
y_minmax = data["y_minmax"]

print("feature_names:", feature_names)
print("X_train:", data["X_train"].shape, "| X_test:", data["X_test"].shape)

prefix_org:resource: 19275 test events with a value not seen in train (encoded as 'unseen')
feature_names: ['Activity', 'time_from_start', 'time_from_previous_event', 'weekday', 'org:resource', 'case:AMOUNT_REQ']
X_train: (101586, 95, 6) | X_test: (27446, 95, 6)


## 3. Model (LSTM, `models.lstm.model`)

Loads the checkpoint from the full experiment sweep.

In [4]:
CHECKPOINT_PATH = os.path.join(
    FULL_SWEEP_EXPERIMENTS_ROOT, DATASET_NAME, "remaining_time_lstm", "model",
    "best_model_remaining_time.keras",
)
assert os.path.exists(CHECKPOINT_PATH), f"{CHECKPOINT_PATH} does not exist yet"

model = tf.keras.models.load_model(CHECKPOINT_PATH)
model.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ prefix (InputLayer) │ (None, 95, 6)     │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cat_embed_Activity  │ (None, 95, 9)     │        225 │ prefix[0][0]      │
│ (CategoricalEmbedd… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cat_embed_weekday   │ (None, 95, 5)     │         45 │ prefix[0][0]      │
│ (CategoricalEmbedd… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cat_embed_org_reso… │ (None, 95, 16)    │      1,040 │ prefix[0][0]      │
│ (CategoricalEmbedd… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ numerical_slice     │ (None, 95, 3)     │          0 │ prefix[0][0]      │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_features     │ (None, 95, 33)    │          0 │ cat_embed_Activi… │
│ (Concatenate)       │                   │            │ cat_embed_weekda… │
│                     │                   │            │ cat_embed_org_re… │
│                     │                   │            │ numerical_slice[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_8 (LSTM)       │ (None, 95, 100)   │     53,600 │ concat_features[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 95, 100)   │        400 │ lstm_8[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_9 (LSTM)       │ (None, 95, 100)   │     80,400 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 95, 100)   │        400 │ lstm_9[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_10 (LSTM)      │ (None, 95, 100)   │     80,400 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 95, 100)   │        400 │ lstm_10[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_11 (LSTM)      │ (None, 95, 100)   │     80,400 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 95, 100)   │        400 │ lstm_11[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_12 (LSTM)      │ (None, 95, 100)   │     80,400 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 95, 100)   │        400 │ lstm_12[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_13 (LSTM)      │ (None, 95, 100)   │     80,400 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 95, 100)   │        400 │ lstm_13[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 1,859,836 (7.09 MB)

 Trainable params: 619,411 (2.36 MB)

 Non-trainable params: 1,600 (6.25 KB)

 Optimizer params: 1,238,825 (4.73 MB)

### Evaluation on Test

In [5]:
y_min, y_max = y_minmax
pred_scaled = model.predict(data["X_test"], verbose=0)[:, 0]
pred_days = (pred_scaled * (y_max - y_min) + y_min) / 86400
actual_days = (data["y_test"] * (y_max - y_min) + y_min) / 86400

mae = np.mean(np.abs(pred_days - actual_days))
corr = np.corrcoef(pred_days, actual_days)[0, 1]
print(f"test MAE: {mae:.2f} days | n_test={len(actual_days)}")

test MAE: 5.29 days | n_test=27446


## 4. Explainability for one prefix, as pandas tables (all 5 unified explainers)

The 5 explainers in `explainability/` (IG, EG, Gradient x Input saliency, SHAP kernel, LIME) run on the SAME demo prefix

In [6]:
column_blocks = build_column_blocks(model_module, categorical_columns, numerical_columns_list,
                                     data["vocab_sizes"], feature_names)
embed_model = model_module.build_embed_model(model)
embed_width = embed_model.output_shape[-1]

lengths = data["test_k"] + 1
DEMO_IDX = int(np.where((lengths >= 8) & (lengths <= 15))[0][0])
demo_length = int(lengths[DEMO_IDX])
demo_prefix = data["X_test"][DEMO_IDX]
print(f"demo prefix: idx={DEMO_IDX}, length={demo_length}")

demo prefix: idx=16, length=8


### Integrated Gradients

In [7]:
zero_baseline_builder = build_ig_zero_baseline(data["max_len"], embed_width)

ig_result = explain_ig(model, model_module, Task.REMAINING_TIME, demo_prefix, demo_length,
                        {"zero": zero_baseline_builder}, column_blocks, feature_names,
                        y_minmax=y_minmax, steps=50)
ig_matrix, ig_completeness_err_pct = ig_result["zero"]

df_ig = pd.DataFrame(ig_matrix, columns=feature_names)
df_ig.index.name = "event_in_prefix"
print(f"IG completeness error: {ig_completeness_err_pct:.4f}%")
df_ig.round(4)

IG completeness error: 0.0001%


,Activity,time_from_start,time_from_previous_event,weekday,org:resource,case:AMOUNT_REQ
event_in_prefix,,,,,,
0,0.0017,0.0000,0.0000,0.0874,0.2348,0.0141
1,0.0828,0.0000,0.0000,0.0906,0.2334,0.0172
2,0.8231,0.0000,-0.0000,0.1037,0.2607,0.0236
3,0.9376,0.0002,-0.0114,0.0672,0.1192,0.0321
4,1.3376,-0.0000,-0.0023,0.0754,0.0810,0.0435
5,1.9218,-0.0005,-0.0004,0.0843,0.3135,0.0606
6,1.7773,-0.0039,-0.1647,-0.1622,0.4543,0.0981
7,3.4977,-0.0072,-0.0004,-0.3442,0.7726,0.1961


### Expected Gradients

In [8]:
eg_background_builder = build_eg_background(data["X_train"], data["train_k"])

eg_matrix, eg_completeness_err_pct, eg_mean_sem, eg_pool_size = explain_eg(
    model, model_module, Task.REMAINING_TIME, demo_prefix, demo_length,
    eg_background_builder, column_blocks, feature_names, y_minmax=y_minmax,
    nsamples=500, batch_size=500, seed=0,
)

df_eg = pd.DataFrame(eg_matrix, columns=feature_names)
df_eg.index.name = "event_in_prefix"
print(f"EG completeness error: {eg_completeness_err_pct:.2f}% (Monte Carlo estimator, noisy at this "
      f"nsamples) | background pool size: {eg_pool_size} | mean SEM: {eg_mean_sem:.4f}")
df_eg.round(4)

EG completeness error: 4.77% (Monte Carlo estimator, noisy at this nsamples) | background pool size: 4002 | mean SEM: 0.0365


,Activity,time_from_start,time_from_previous_event,weekday,org:resource,case:AMOUNT_REQ
event_in_prefix,,,,,,
0,0.0000,0.0000,0.0000,0.0126,0.0000,-0.0007
1,0.0000,0.0000,0.0000,0.0138,0.0000,-0.0011
2,0.0216,0.0000,0.0008,0.0269,-0.0073,-0.0016
3,0.2965,-0.0001,-0.0120,0.0248,-0.0261,-0.0020
4,0.2840,-0.0002,0.0022,0.0353,-0.0897,-0.0020
5,0.4216,-0.0002,0.0055,0.0471,0.0566,-0.0018
6,-0.0924,-0.0028,-0.1192,-0.1087,0.1188,-0.0020
7,0.6506,-0.0032,0.0122,-0.1532,0.1834,-0.0016


### Gradient x Input saliency

Signed and abs

In [9]:
grad_signed, grad_abs = explain_gradient_saliency(
    model, model_module, Task.REMAINING_TIME, demo_prefix, demo_length, column_blocks, feature_names,
    y_minmax=y_minmax,
)

df_grad_signed = pd.DataFrame(grad_signed, columns=feature_names)
df_grad_signed.index.name = "event_in_prefix"
print("Gradient x Input (signed):")
df_grad_signed.round(4)

Gradient x Input (signed):


,Activity,time_from_start,time_from_previous_event,weekday,org:resource,case:AMOUNT_REQ
event_in_prefix,,,,,,
0,-0.0051,0.0000,0.0000,0.0485,0.1326,0.0123
1,0.0646,0.0000,0.0000,0.0417,0.0920,0.0167
2,1.1016,0.0000,-0.0000,0.0605,0.0749,0.0346
3,1.2299,-0.0001,-0.0248,0.0508,0.0937,0.0443
4,1.4679,-0.0004,-0.0039,0.0533,0.0367,0.0499
5,1.6136,-0.0008,-0.0004,0.0494,0.2325,0.0539
6,1.3380,-0.0044,-0.1608,-0.1562,0.3473,0.0768
7,1.8728,-0.0043,-0.0002,-0.1867,0.4492,0.1091


In [10]:
df_grad_abs = pd.DataFrame(grad_abs, columns=feature_names)
df_grad_abs.index.name = "event_in_prefix"
print("Gradient x Input (absolute value):")
df_grad_abs.round(4)

Gradient x Input (absolute value):


,Activity,time_from_start,time_from_previous_event,weekday,org:resource,case:AMOUNT_REQ
event_in_prefix,,,,,,
0,0.0367,0.0000,0.0000,0.0485,0.1684,0.0123
1,0.0822,0.0000,0.0000,0.0522,0.2008,0.0167
2,1.1033,0.0000,0.0000,0.0847,0.4026,0.0346
3,1.2299,0.0001,0.0248,0.0677,0.2303,0.0443
4,1.4679,0.0004,0.0039,0.0760,0.3042,0.0499
5,1.6136,0.0008,0.0004,0.0781,0.3702,0.0539
6,1.3380,0.0044,0.1608,0.2360,0.4313,0.0768
7,1.9156,0.0043,0.0002,0.2868,0.5582,0.1091


### SHAP (KernelExplainer)

In [11]:
shap_matrix, shap_gap = explain_shap(
    model, model_module, Task.REMAINING_TIME, demo_prefix, demo_length, feature_names,
    y_minmax=y_minmax, nsamples=1000, seed=0,
)

df_shap = pd.DataFrame(shap_matrix, columns=feature_names)
df_shap.index.name = "event_in_prefix"
print(f"SHAP completeness check: sum(shap) = {shap_matrix.sum():.4f} vs. gap f(x)-f(baseline) = {shap_gap:.4f}")
df_shap.round(4)

100%|██████████| 1/1 [00:04<00:00,  4.67s/it]

SHAP completeness check: sum(shap) = 13.1458 vs. gap f(x)-f(baseline) = 13.1458


,Activity,time_from_start,time_from_previous_event,weekday,org:resource,case:AMOUNT_REQ
event_in_prefix,,,,,,
0,0.0444,0.0000,0.0000,0.0871,0.2877,-0.0037
1,0.0938,0.0000,0.0000,0.1073,0.0645,0.1028
2,0.6804,0.0995,-0.0515,-0.0526,0.2987,0.1144
3,0.8373,0.0704,0.0597,0.0493,0.0829,-0.0260
4,1.1799,-0.0511,-0.0022,0.0650,0.0279,-0.0276
5,1.7224,-0.1064,0.0082,0.0440,0.3731,0.1104
6,1.6820,0.0964,0.0170,-0.0318,0.4255,0.1764
7,4.2093,0.0253,-0.1515,-0.3056,0.5595,0.1535


### LIME

In [12]:
lime_col_index, lime_event_vars, lime_case_vars, lime_get_layout = build_lime_layout(
    feature_names, columns_info, categorical_columns, numerical_columns_list)
lime_background_builder = build_lime_background(data["X_train"], data["train_k"])

lime_matrix = explain_lime(
    model, model_module, Task.REMAINING_TIME, demo_prefix, demo_length, lime_background_builder,
    lime_col_index, lime_event_vars, lime_case_vars, lime_get_layout, feature_names,
    y_minmax=y_minmax, num_samples=500, seed=0,
)

df_lime = pd.DataFrame(lime_matrix, columns=feature_names)
df_lime.index.name = "event_in_prefix"
df_lime.round(4)

,Activity,time_from_start,time_from_previous_event,weekday,org:resource,case:AMOUNT_REQ
event_in_prefix,,,,,,
0,0.0000,0.0000,0.0000,0.1299,0.0000,-0.4893
1,0.0000,-0.0101,0.2432,0.3295,0.0000,-0.4893
2,0.5455,-0.4033,-0.0848,-0.0049,0.0551,-0.4893
3,0.3866,0.2324,-0.1608,0.4951,0.2387,-0.4893
4,0.6643,0.0910,0.2156,-0.1291,0.3967,-0.4893
5,0.5934,-0.1626,-0.0812,0.0550,-0.0829,-0.4893
6,-0.0517,-0.4463,-0.2986,-0.1504,0.7782,-0.4893
7,0.6598,0.0936,-0.1095,-0.3628,-0.2010,-0.4893


## 5. Sensitivity study (RIS/ROS/FA/RA) over a sample of prefixes

Uses the perturbation system in `sensitivity_evaluation.sensitivity_perturbations` (Gaussian noise
on numerical features, RRF sampling -- frequency + embedding distance -- on categorical features)
and the Monte Carlo routine in `sensitivity_evaluation.metrics` (OpenXAI-style ratio-of-norms
RIS/ROS, plus Feature/Rank Agreement), explaining each perturbation with Integrated Gradients (the
`explain_fn` passed to `metrics` is `explain_ig` from above, wrapped in the 4-argument signature
`metrics` expects -- see the cell below).

In [13]:
def explain_fn_ig(model, x, length, feature_names):
    matrix, _ = explain_ig(model, model_module, Task.REMAINING_TIME, x, length,
                            {"zero": zero_baseline_builder}, column_blocks, feature_names,
                            y_minmax=y_minmax, steps=50)["zero"]
    return matrix


def h_fn_batched(model, x_batch):
    return model.predict(x_batch, verbose=0)[:, 0]


embeddings_by_var = {name: extract_embedding(model, name) for name in categorical_columns}
sampling_distributions_by_var = build_all_categorical_sampling_distributions(
    data["X_train"], data["train_k"], categorical_columns, columns_info, embeddings_by_var)

numerical_names_in_order = [feature_names[c] for c in numerical_columns_list]
numerical_columns = dict(zip(numerical_names_in_order, numerical_columns_list))

gap_columns = {"time_from_previous_event": [("time_from_start", None)]}

In [14]:
N_SAMPLES_DEMO = 5
N_MONTECARLO = 15

sample_idx = [int(i) for i in np.where((lengths >= 6) & (lengths <= 15))[0][:N_SAMPLES_DEMO]]
print("sample prefixes:", sample_idx)

rng = np.random.default_rng(0)
rows = []
for idx in sample_idx:
    prefix = data["X_test"][idx]
    length = int(lengths[idx])
    result = relative_stability_montecarlo_v4_with_agreement(
        model=model, prefix=prefix, length=length, explain_fn=explain_fn_ig, h_fn=h_fn_batched,
        feature_names=feature_names, numerical_columns=numerical_columns,
        categorical_columns=categorical_columns, columns_info=columns_info,
        numerical_minmax=numerical_minmax, gap_columns=gap_columns, implicit_cumulative_columns={},
        percent_cells_numeric=0.3, noise_scale=0.3, min_noise=1.0,
        sampling_distributions_by_var=sampling_distributions_by_var, percent_cells_categorical=0.3,
        embeddings_by_var=embeddings_by_var, n_samples=N_MONTECARLO, rng=rng, y_minmax=y_minmax,
        k_agreement=3,
    )
    rows.append({
        "idx": idx, "length": length,
        "RIS_max": result["RIS"]["max"], "RIS_mean": result["RIS"]["mean"], "RIS_median": result["RIS"]["median"],
        "ROS_max": result["ROS"]["max"], "ROS_mean": result["ROS"]["mean"], "ROS_median": result["ROS"]["median"],
        "FA_min": result["FA"]["min"], "FA_mean": result["FA"]["mean"], "FA_median": result["FA"]["median"],
        "RA_min": result["RA"]["min"], "RA_mean": result["RA"]["mean"], "RA_median": result["RA"]["median"],
    })

sensitivity_df = pd.DataFrame(rows)
sensitivity_df.round(4)

sample prefixes: [14, 15, 16, 17, 18]


,idx,length,RIS_max,RIS_mean,RIS_median,ROS_max,ROS_mean,ROS_median,FA_min,FA_mean,FA_median,RA_min,RA_mean,RA_median
0,14,6,2.0590,0.9155,0.7382,10.8979,3.7441,3.7817,0.6667,0.8000,0.6667,0.0,0.3778,0.3333
1,15,7,1.9283,0.8464,0.7938,5.4512,2.8457,2.3441,0.6667,0.8444,1.0000,0.0,0.4222,0.3333
2,16,8,1.6547,0.7272,0.6691,12.2250,4.1087,3.2888,0.6667,0.8667,1.0000,0.0,0.4000,0.3333
3,17,9,1.4821,0.9021,0.8997,8.5159,3.7896,3.4369,0.6667,0.8222,0.6667,0.0,0.4000,0.3333
4,18,10,1.0837,0.6682,0.7168,29.9788,8.5349,4.7852,0.6667,0.7778,0.6667,0.0,0.5333,0.3333


- **RIS/ROS**: ratio-of-norms
- **FA/RA**: Feature Agreement / Rank Agreement between the original explanation and each perturbed one (1 = identical top-k, 0 = no overlap)

This is just a demo over 5 prefixes with 15 perturbations each. The real study would use many more prefixes and a larger `n_samples`.